In [1]:
from pathlib import Path
import sys

# ---- Project root setup ----

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

# ---- Set plan filename ----

from spc_agent.agent.agent_runner import ask_agent

In [2]:
print('---------------')
print('planner_prompt.py')
print('---------------')

prompt_text = Path(PROJECT_ROOT / "spc_agent/agent/planner_prompt.py").read_text()
print(prompt_text)

---------------
planner_prompt.py
---------------
from __future__ import annotations
from textwrap import dedent
from pathlib import Path

schema_text = """
You are generating JSON plans for the Deterministic SPC Agent.
Output JSON only.

-----
Supported plans
-----
1. Execution plan - performs a database query and plotting workflow from a new prompt. Consists of one or more run objects.
2. Replot plan - references a previous run and/or job to specify new output objects generated from existing artifacts

-----
Execution plans
-----

For ask_agent(), prefer a single execution run object, not a plan library, unless multiple runs are explicitly required.

Single run shape:
- run_id: string. Brief summary of the user prompt in snake case
- request_text: string. Stores user prompt
- jobs: non-empty list

-----
Supported queries
-----

1. Maintenance/Repair/PM History

- Examples of PM history execution prompts:
  - Show PM history for <entity(s)> for the last <time range>
  - What is the re

## Improved analytical workflows

In [3]:
# PM history
result = ask_agent(
    prompt="Show the maintenance history on CPR.",
    project_root=PROJECT_ROOT,
    planner_backend="auto",
    verbose=True,
)

=== ask_agent: prompt ===
Show the maintenance history on CPR.

=== planner configuration ===
planner_backend: auto
planner_file: planner/demo_gallery.json
planner_config: {}

=== planner result summary ===
planner_backend: llm
planner_context: llm_generated_plan

=== planner raw output ===
{
  "runs": [
    {
      "run_id": "cpr_pm_history",
      "request_text": "Show the maintenance history on CPR.",
      "jobs": [
        {
          "job_id": "cpr_pm_history",
          "sql_template": "pm_event_sensor_history",
          "preprocess": "pm_event_ooc_summary",
          "filters": {
            "entity_group": "CPR",
            "entity": null,
            "sensor": null,
            "start_ts": null,
            "end_ts": null
          },
          "outputs": {
            "tables": [
              {
                "table": "pm_event_summary_table",
                "table_name": "cpr_pm_history.csv"
              }
            ]
          }
        }
      ]
    }
  ]
}

=== p

In [4]:
# Review data leading up to PM, known cause
result = ask_agent(
    prompt="Plot vibration on CPR15 leading up to the work on 1/9.",
    project_root=PROJECT_ROOT,
    planner_backend="auto",
    verbose=True,
)

=== ask_agent: prompt ===
Plot vibration on CPR15 leading up to the work on 1/9.

=== planner configuration ===
planner_backend: auto
planner_file: planner/demo_gallery.json
planner_config: {}

=== planner result summary ===
planner_backend: llm
planner_context: llm_generated_plan

=== planner raw output ===
{
  "runs": [
    {
      "run_id": "cpr15_vibration_pre_maintenance",
      "request_text": "Plot vibration on CPR15 leading up to the work on 1/9.",
      "jobs": [
        {
          "job_id": "cpr15_vibration_pre_maintenance",
          "sql_template": "entity_sensor_history",
          "preprocess": "ewma_spc",
          "filters": {
            "entity_group": "CPR",
            "entity": "CPR15",
            "sensor": "vibration_rms",
            "start_ts": null,
            "end_ts": "2024-01-09T00:00:00"
          },
          "outputs": {
            "plots": [
              {
                "plot": "spc_time_series",
                "plot_name": "cpr15_vibration_pre_m

In [5]:
# Review data leading up to PM, unknown cause
result = ask_agent(
    prompt="Plot all sensors on CPR15 for 1/8-1/12.",
    project_root=PROJECT_ROOT,
    planner_backend="auto",
    verbose=True,
)

=== ask_agent: prompt ===
Plot all sensors on CPR15 for 1/8-1/12.

=== planner configuration ===
planner_backend: auto
planner_file: planner/demo_gallery.json
planner_config: {}

=== planner result summary ===
planner_backend: llm
planner_context: llm_generated_plan

=== planner raw output ===
{
  "runs": [
    {
      "run_id": "cpr15_all_sensors_2024_01_08_2024_01_12",
      "request_text": "Plot all sensors on CPR15 for 1/8-1/12.",
      "jobs": [
        {
          "job_id": "cpr15_ambient_temp",
          "sql_template": "entity_sensor_history",
          "preprocess": "ewma_spc",
          "filters": {
            "entity_group": "CPR",
            "entity": "CPR15",
            "sensor": "ambient_temp",
            "start_ts": "2024-01-08T00:00:00",
            "end_ts": "2024-01-12T23:59:59"
          },
          "outputs": {
            "plots": [
              {
                "plot": "spc_time_series",
                "plot_name": "cpr15_ambient_temp_2024_01_08_2024_01_12

In [6]:
# Health check
result = ask_agent(
    prompt="How is the tool doing now?",
    project_root=PROJECT_ROOT,
    planner_backend="auto",
    verbose=True,
)

=== ask_agent: prompt ===
How is the tool doing now?

=== planner configuration ===
planner_backend: auto
planner_file: planner/demo_gallery.json
planner_config: {}

=== planner result summary ===
planner_backend: llm
planner_context: llm_generated_plan

=== planner raw output ===
{
  "mode": "replot",
  "run_ref": "latest"
}

=== parsed planner plan ===
{
  "mode": "replot",
  "run_ref": "latest"
}

=== recovery sentinel detected ===
Planner requested previous-run context for a second planning pass.

=== recovery planner raw output ===
{
  "runs": [
    {
      "run_id": "cpr15_health_summary_now",
      "request_text": "How is the tool doing now?",
      "jobs": [
        {
          "job_id": "cpr15_health_summary",
          "sql_template": "entity_all_sensor_history",
          "preprocess": "multi_sensor_health_summary",
          "filters": {
            "entity_group": "CPR",
            "entity": "CPR15",
            "sensor": null,
            "start_ts": null,
            "e

/Users/mcmoore/anaconda_projects/GitHub/agentic-predictive-maintenance/preprocess/multi_sensor_health_summary.py:164: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_per_sensor)


In [ ]:
# Health check
result = ask_agent(
    prompt="Show the plot for ambient temp",
    project_root=PROJECT_ROOT,
    planner_backend="auto",
    verbose=True,
)